In [0]:
%sql
-- dim_drivers
COMMENT ON TABLE formula1_incr.gold.dim_drivers IS 'Driver dimension: one row per Formula 1 driver with name, date of birth, nationality and nationality region. Primary key: driver_id.';

ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN driver_id          COMMENT 'Unique driver identifier (primary key). Joins to fact_session_results.driver_id.';
ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN driver_name        COMMENT 'Full name of the driver.';
ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN date_of_birth      COMMENT 'Driver date of birth.';
ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN nationality        COMMENT 'Driver nationality (e.g. British, Dutch).';
ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN nationality_region COMMENT 'Geographic region derived from the driver nationality via ref_nationality_region.';
ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN created_timestamp  COMMENT 'Timestamp when the record was first loaded into the gold layer.';
ALTER TABLE formula1_incr.gold.dim_drivers ALTER COLUMN updated_timestamp  COMMENT 'Timestamp when the record was last updated in the gold layer.';

In [0]:
%sql
-- dim_constructors
COMMENT ON TABLE formula1_incr.gold.dim_constructors IS 'Constructor (team) dimension: one row per Formula 1 constructor with name, nationality and nationality region. Primary key: constructor_id.';

ALTER TABLE formula1_incr.gold.dim_constructors ALTER COLUMN constructor_id     COMMENT 'Unique constructor identifier (primary key). Joins to fact_session_results.constructor_id.';
ALTER TABLE formula1_incr.gold.dim_constructors ALTER COLUMN constructor_name   COMMENT 'Constructor (team) name.';
ALTER TABLE formula1_incr.gold.dim_constructors ALTER COLUMN nationality        COMMENT 'Constructor nationality. Foreign key to ref_nationality_region.nationality.';
ALTER TABLE formula1_incr.gold.dim_constructors ALTER COLUMN nationality_region COMMENT 'Geographic region derived from the constructor nationality via ref_nationality_region.';
ALTER TABLE formula1_incr.gold.dim_constructors ALTER COLUMN created_timestamp  COMMENT 'Timestamp when the record was first loaded into the gold layer.';
ALTER TABLE formula1_incr.gold.dim_constructors ALTER COLUMN updated_timestamp  COMMENT 'Timestamp when the record was last updated in the gold layer.';

In [0]:
%sql
-- dim_races
COMMENT ON TABLE formula1_incr.gold.dim_races IS 'Race (Grand Prix event) dimension: one row per season and round with race name, date and circuit location. Primary key: (season, round).';

ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN season            COMMENT 'Championship season year. Part of the primary key (season, round).';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN round             COMMENT 'Round number of the race within the season. Part of the primary key (season, round).';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN race_name         COMMENT 'Official Grand Prix name.';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN race_date         COMMENT 'Date of the race.';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN circuit_name      COMMENT 'Name of the circuit where the race was held.';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN locality          COMMENT 'City or locality of the circuit.';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN country           COMMENT 'Country of the circuit.';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN created_timestamp COMMENT 'Timestamp when the record was first loaded into the gold layer.';
ALTER TABLE formula1_incr.gold.dim_races ALTER COLUMN updated_timestamp COMMENT 'Timestamp when the record was last updated in the gold layer.';

In [0]:
%sql
-- fact_session_results
COMMENT ON TABLE formula1_incr.gold.fact_session_results IS 'Fact table of session results: one row per driver per session (RACE or SPRINT) per race. Primary key: (season, round, driver_id, session_type). Foreign keys to dim_races, dim_drivers and dim_constructors.';

ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN season              COMMENT 'Championship season year. Foreign key to dim_races (season, round).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN round               COMMENT 'Round number within the season. Foreign key to dim_races (season, round).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN constructor_id      COMMENT 'Constructor the driver raced for. Foreign key to dim_constructors.constructor_id.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN driver_id           COMMENT 'Driver identifier. Foreign key to dim_drivers.driver_id.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN grid_position       COMMENT 'Starting grid position of the driver.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN completed_laps      COMMENT 'Number of laps completed by the driver in the session.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN car_number          COMMENT 'Car number used by the driver.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN points              COMMENT 'Championship points earned in the session.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN final_position      COMMENT 'Classified finishing position (1 = winner). NULL when the driver was not classified.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN final_position_text COMMENT 'Finishing position as text, including non-numeric codes (e.g. R = retired, D = disqualified).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN status              COMMENT 'Result status of the driver (e.g. Finished, +1 Lap, Accident).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN session_type        COMMENT 'Session type: RACE or SPRINT. Part of the primary key.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN is_win              COMMENT 'True when the driver won the session (final_position = 1).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN is_podium           COMMENT 'True when the driver finished in the top 3 (final_position <= 3).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN has_points          COMMENT 'True when the driver scored championship points (points > 0).';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN created_timestamp   COMMENT 'Timestamp when the record was first loaded into the gold layer.';
ALTER TABLE formula1_incr.gold.fact_session_results ALTER COLUMN updated_timestamp   COMMENT 'Timestamp when the record was last updated in the gold layer.';

In [0]:
%sql
-- ref_nationality_region
COMMENT ON TABLE formula1_incr.gold.ref_nationality_region IS 'Reference mapping from nationality to geographic region. Used to enrich drivers and constructors. Primary key: nationality.';

ALTER TABLE formula1_incr.gold.ref_nationality_region ALTER COLUMN nationality COMMENT 'Nationality value (primary key), e.g. British, German.';
ALTER TABLE formula1_incr.gold.ref_nationality_region ALTER COLUMN region      COMMENT 'Geographic region the nationality belongs to (e.g. Europe, Americas).';

In [0]:
%sql
-- Check table and column comments after running the cells above
SELECT t.table_name, t.comment AS table_comment, c.column_name, c.comment AS column_comment
FROM formula1_incr.information_schema.tables t
JOIN formula1_incr.information_schema.columns c
  ON c.table_schema = t.table_schema AND c.table_name = t.table_name
WHERE t.table_schema = 'gold'
  AND c.table_schema = 'gold'
  AND t.table_type = 'MANAGED'
ORDER BY t.table_name, c.ordinal_position;

In [0]:
tables = spark.sql("SHOW TABLES IN formula1_incr.gold").collect()

for table in tables:
    table_name = table["tableName"]
    print(f"\n{'='*80}")
    print(f"TABLE: formula1_incr.gold.{table_name}")
    print(f"{'='*80}")
    spark.sql(f"DESCRIBE TABLE EXTENDED formula1_incr.gold.{table_name}").show(100, truncate=False)